# OptiML support-learning results analysis

This notebook loads the `.npz` files produced by `train_baseline.py` and creates visualizations for comparing GD, SGD, Adam/AdamW, Muon, and weight decay.

The main question is: **which optimizer suppresses irrelevant input directions most strongly while fitting the target?**

Expected output structure:

```text
outputs/
└── linear/
    ├── linear_gd_lr0.1_200000iters.npz
    ├── linear_gd_lr0.1_200000iters_config.json
    ├── ...
```

Change `RESULTS_DIR` below if needed.


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Change this to your folder if the notebook is not launched from ~/Desktop/optiML_support
RESULTS_DIR = Path("outputs/linear")

assert RESULTS_DIR.exists(), f"Could not find {RESULTS_DIR.resolve()}"


## 1. Load all runs

Each run consists of:

- one `.npz` file with arrays,
- one `_config.json` file with hyperparameters.

The script saves arrays like:

```text
traj0_layer0_init
traj0_layer0_post
traj0_losses
traj0_irelnorms
```


In [ ]:
def load_run(npz_path: Path) -> dict:
    config_path = npz_path.with_name(npz_path.stem + "_config.json")
    with open(config_path) as f:
        config = json.load(f)

    data = np.load(npz_path, allow_pickle=True)

    traj_ids = sorted({
        int(m.group(1))
        for key in data.files
        if (m := re.match(r"traj(\d+)_losses", key))
    })

    run = {
        "name": npz_path.stem,
        "npz_path": npz_path,
        "config": config,
        "data": data,
        "traj_ids": traj_ids,
    }
    return run


runs = [load_run(p) for p in sorted(RESULTS_DIR.glob("*.npz"))]
len(runs), [r["name"] for r in runs]


In [ ]:
def run_label(run: dict) -> str:
    c = run["config"]
    opt = c.get("optimizer", "?")
    lr = c.get("lr", "?")
    bs = "full" if opt == "gd" else c.get("batch_size", "?")
    wd = c.get("weight_decay", 0.0)
    if wd:
        return f"{opt}, bs={bs}, lr={lr}, wd={wd}"
    return f"{opt}, bs={bs}, lr={lr}"


summary_rows = []
for run in runs:
    c = run["config"]
    data = run["data"]

    for traj in run["traj_ids"]:
        losses = data[f"traj{traj}_losses"]
        irelnorms = data[f"traj{traj}_irelnorms"]

        summary_rows.append({
            "run": run["name"],
            "label": run_label(run),
            "traj": traj,
            "optimizer": c.get("optimizer"),
            "batch_size": "full" if c.get("optimizer") == "gd" else c.get("batch_size"),
            "lr": c.get("lr"),
            "weight_decay": c.get("weight_decay"),
            "target": c.get("target"),
            "n_iters": c.get("n_iters"),
            "final_loss": float(losses[-1]),
            "min_loss": float(losses.min()),
            "final_irelnorm": float(irelnorms[-1]),
            "min_irelnorm": float(irelnorms.min()),
            "init_irelnorm": float(irelnorms[0]),
        })

summary = pd.DataFrame(summary_rows)
summary.sort_values(["final_irelnorm", "final_loss"])


## 2. Loss curves

Use log scale because optimizer differences can be huge and early training often dominates the visual impression.


In [ ]:
def plot_metric_curves(metric: str, ylabel: str, logy: bool = False, max_points: int = 2000):
    plt.figure(figsize=(10, 6))

    for run in runs:
        data = run["data"]
        label = run_label(run)

        curves = []
        for traj in run["traj_ids"]:
            y = data[f"traj{traj}_{metric}"]
            curves.append(y)

        # Average across trajectories if multiple exist.
        min_len = min(len(c) for c in curves)
        curves = np.stack([c[:min_len] for c in curves])
        mean = curves.mean(axis=0)
        std = curves.std(axis=0)

        # Downsample for readability.
        idx = np.linspace(0, min_len - 1, min(max_points, min_len)).astype(int)
        x = idx
        y = mean[idx]
        s = std[idx]

        plt.plot(x, y, label=label)
        if len(run["traj_ids"]) > 1:
            plt.fill_between(x, y - s, y + s, alpha=0.15)

    plt.xlabel("gradient step")
    plt.ylabel(ylabel)
    if logy:
        plt.yscale("log")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_metric_curves("losses", "MSE loss", logy=True)


## 3. Irrelevant-feature norm curves

This is probably the most important plot for the project.

The training script tracks:

```python
np.linalg.norm(W1[:, r:d])
```

where `W1` is the first-layer weight matrix. Lower means the network is using irrelevant input coordinates less.


In [ ]:
plot_metric_curves("irelnorms", r"$\|W_1[:, r:d]\|_F$", logy=False)


## 4. Final score table

This table helps you quickly see which optimizer finished with low loss and low irrelevant norm.


In [ ]:
display(
    summary
    .groupby(["label", "optimizer", "batch_size", "weight_decay"], as_index=False)
    .agg(
        final_loss_mean=("final_loss", "mean"),
        final_loss_std=("final_loss", "std"),
        final_irelnorm_mean=("final_irelnorm", "mean"),
        final_irelnorm_std=("final_irelnorm", "std"),
        min_irelnorm_mean=("min_irelnorm", "mean"),
    )
    .sort_values("final_irelnorm_mean")
)


## 5. Scatter: final loss vs final irrelevant norm

The ideal region is bottom-left: low loss and low irrelevant-feature norm.

A method that gets low loss but high irrelevant norm fits the data while still relying on irrelevant directions.


In [ ]:
plt.figure(figsize=(8, 6))

agg = (
    summary
    .groupby("label", as_index=False)
    .agg(final_loss=("final_loss", "mean"),
         final_irelnorm=("final_irelnorm", "mean"))
)

for _, row in agg.iterrows():
    plt.scatter(row["final_irelnorm"], row["final_loss"], s=80)
    plt.annotate(row["label"], (row["final_irelnorm"], row["final_loss"]),
                 xytext=(5, 5), textcoords="offset points", fontsize=9)

plt.xlabel(r"final $\|W_1[:, r:d]\|_F$")
plt.ylabel("final MSE loss")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Relevant vs irrelevant first-layer weight norms

For each final model, compare the norm of the first-layer columns corresponding to relevant features versus irrelevant features.

This is a more interpretable version of the support-learning diagnostic.


In [ ]:
def first_layer_split_norms(run: dict, traj: int = 0, state: str = "post") -> tuple[float, float]:
    c = run["config"]
    r = c["r"]
    d = c["d"]
    W1 = run["data"][f"traj{traj}_layer0_{state}"]
    relevant = np.linalg.norm(W1[:, :r])
    irrelevant = np.linalg.norm(W1[:, r:d])
    return float(relevant), float(irrelevant)


rows = []
for run in runs:
    for traj in run["traj_ids"]:
        rel_init, irrel_init = first_layer_split_norms(run, traj, "init")
        rel_post, irrel_post = first_layer_split_norms(run, traj, "post")
        rows.append({
            "label": run_label(run),
            "traj": traj,
            "rel_init": rel_init,
            "irrel_init": irrel_init,
            "rel_post": rel_post,
            "irrel_post": irrel_post,
            "ratio_post_irrel_over_rel": irrel_post / rel_post if rel_post > 0 else np.nan,
        })

split_df = pd.DataFrame(rows)
display(split_df.sort_values("ratio_post_irrel_over_rel"))


In [ ]:
plot_df = (
    split_df
    .groupby("label", as_index=False)
    .agg(rel_post=("rel_post", "mean"),
         irrel_post=("irrel_post", "mean"))
)

x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(12, 6))
plt.bar(x - width/2, plot_df["rel_post"], width, label="relevant")
plt.bar(x + width/2, plot_df["irrel_post"], width, label="irrelevant")
plt.xticks(x, plot_df["label"], rotation=45, ha="right")
plt.ylabel("first-layer Frobenius norm")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 7. First-layer heatmaps

These show which input coordinates are used by each hidden unit.

Columns `0` to `r-1` are relevant. Columns `r` to `d-1` are irrelevant.


In [ ]:
def plot_first_layer_heatmaps(state: str = "post", traj: int = 0):
    for run in runs:
        c = run["config"]
        r = c["r"]
        W1 = run["data"][f"traj{traj}_layer0_{state}"]

        plt.figure(figsize=(8, 4))
        plt.imshow(W1, aspect="auto")
        plt.colorbar(label="weight")
        plt.axvline(r - 0.5, linestyle="--", linewidth=2)
        plt.title(f"{run_label(run)} — layer 0 {state}")
        plt.xlabel("input coordinate")
        plt.ylabel("hidden unit")
        plt.tight_layout()
        plt.show()


plot_first_layer_heatmaps("post", traj=0)


## 8. Singular values of layer matrices

This is useful for the spectral angle of the project: different optimizers may produce different rank structure or implicit compression.

For each layer, we plot singular values at initialization and after training.


In [ ]:
def layer_ids(run: dict) -> list[int]:
    ids = []
    for key in run["data"].files:
        m = re.match(r"traj0_layer(\d+)_post", key)
        if m:
            ids.append(int(m.group(1)))
    return sorted(ids)


def plot_singular_values(traj: int = 0):
    for run in runs:
        ids = layer_ids(run)
        for layer in ids:
            W_init = run["data"][f"traj{traj}_layer{layer}_init"]
            W_post = run["data"][f"traj{traj}_layer{layer}_post"]

            s_init = np.linalg.svd(W_init, compute_uv=False)
            s_post = np.linalg.svd(W_post, compute_uv=False)

            plt.figure(figsize=(7, 4))
            plt.plot(np.arange(1, len(s_init) + 1), s_init, marker="o", label="init")
            plt.plot(np.arange(1, len(s_post) + 1), s_post, marker="o", label="post")
            plt.title(f"{run_label(run)} — layer {layer}")
            plt.xlabel("singular value index")
            plt.ylabel("singular value")
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()


plot_singular_values(traj=0)


## 9. Effective rank

A compact spectral summary. Effective rank is high when singular values are spread out, and low when only a few singular directions dominate.


In [ ]:
def effective_rank(W: np.ndarray, eps: float = 1e-12) -> float:
    s = np.linalg.svd(W, compute_uv=False)
    p = s / (s.sum() + eps)
    entropy = -(p * np.log(p + eps)).sum()
    return float(np.exp(entropy))


rank_rows = []
for run in runs:
    for traj in run["traj_ids"]:
        for layer in layer_ids(run):
            W_init = run["data"][f"traj{traj}_layer{layer}_init"]
            W_post = run["data"][f"traj{traj}_layer{layer}_post"]
            rank_rows.append({
                "label": run_label(run),
                "traj": traj,
                "layer": layer,
                "eff_rank_init": effective_rank(W_init),
                "eff_rank_post": effective_rank(W_post),
                "delta_eff_rank": effective_rank(W_post) - effective_rank(W_init),
            })

rank_df = pd.DataFrame(rank_rows)
display(rank_df.sort_values(["layer", "eff_rank_post"]))


In [ ]:
for layer in sorted(rank_df["layer"].unique()):
    layer_df = (
        rank_df[rank_df["layer"] == layer]
        .groupby("label", as_index=False)
        .agg(eff_rank_post=("eff_rank_post", "mean"))
        .sort_values("eff_rank_post")
    )

    plt.figure(figsize=(10, 4))
    plt.bar(layer_df["label"], layer_df["eff_rank_post"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("effective rank after training")
    plt.title(f"Layer {layer}")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


## 10. Optional: compare only selected runs

Useful when the all-runs plot is too cluttered.


In [ ]:
# Edit these substrings to filter runs.
SELECT = ["gd", "sgd32", "sgd512", "adam", "adamw", "muon"]

selected_runs = [
    run for run in runs
    if any(s in run["name"] for s in SELECT)
]

old_runs = runs
runs = selected_runs

plot_metric_curves("losses", "MSE loss", logy=True)
plot_metric_curves("irelnorms", r"$\|W_1[:, r:d]\|_F$", logy=False)

runs = old_runs


## Things to discuss from these plots

Good discussion questions:

1. Which methods get low loss?
2. Among low-loss methods, which get low irrelevant norm?
3. Does minibatch SGD beat full-batch GD on support selection?
4. Does weight decay mimic SGD's implicit regularization?
5. Do Adam/AdamW/Muon suppress irrelevant directions or just fit?
6. Are spectral changes correlated with support selection?
7. Does effective rank shrink more for some optimizers?


# Paper-style figures

This section reproduces plots in the style of the paper:

- **Figure 1 style:** first-layer weight matrices and Gram matrices at initialization vs convergence.
- **Figure 2 style:** eigenvalue histogram for the first-layer weight Gram matrix.
- **Figure 3 style:** norm of irrelevant first-layer weights over training time.

In the paper, the key visual story is that all methods can reach similar loss, but smaller-batch SGD more clearly removes the irrelevant input directions from the first layer.


## Helper functions for paper-style labels and ordering

In [ ]:
def short_paper_label(run: dict) -> str:
    c = run["config"]
    opt = c.get("optimizer")
    bs = c.get("batch_size")
    wd = c.get("weight_decay", 0.0)

    if opt == "gd" and wd:
        return f"GD + wd"
    if opt == "gd":
        return "GD"
    if opt == "sgd":
        return f"SGD-{bs}"
    if opt == "adam":
        return f"Adam-{bs}"
    if opt == "adamw":
        return f"AdamW-{bs}"
    if opt == "muon":
        return f"Muon-{bs}"
    return run["name"]


def paper_run_order_key(run: dict):
    c = run["config"]
    opt = c.get("optimizer")
    bs = c.get("batch_size", 10**9)
    wd = c.get("weight_decay", 0.0)

    if opt == "gd" and not wd:
        return (0, 0)
    if opt == "sgd":
        return (1, bs)
    if opt == "gd" and wd:
        return (2, 0)
    if opt == "adam":
        return (3, bs)
    if opt == "adamw":
        return (4, bs)
    if opt == "muon":
        return (5, bs)
    return (9, 0)


paper_runs = sorted(runs, key=paper_run_order_key)
[short_paper_label(r) for r in paper_runs]


## Figure 1 style: first-layer weights and Gram matrices

The paper shows, for each optimizer:

1. the first-layer matrix \(W_1\),
2. the Gram matrix \(W_1^T W_1\).

Why \(W_1^T W_1\)? Because columns of \(W_1\) correspond to input coordinates. So this Gram matrix compares input-feature directions after the first layer. If irrelevant coordinates are suppressed, the rows/columns after the relevant block should become visually small.

Here we make two versions:

- init-only baseline,
- final matrices for every run.


In [ ]:
def get_W1(run: dict, state: str = "post", traj: int = 0) -> np.ndarray:
    return run["data"][f"traj{traj}_layer0_{state}"]


def gram_input_space(W1: np.ndarray) -> np.ndarray:
    # W1 shape is usually hidden_width x input_dim.
    # W1.T @ W1 gives a d x d matrix comparing input coordinates.
    return W1.T @ W1


def final_loss_for_run(run: dict, traj: int = 0) -> float:
    losses = run["data"][f"traj{traj}_losses"]
    return float(losses[-1])


def symmetric_abs_limit(mats):
    vmax = max(float(np.max(np.abs(M))) for M in mats)
    return vmax if vmax > 0 else 1.0


def positive_limit(mats):
    vmax = max(float(np.max(M)) for M in mats)
    return vmax if vmax > 0 else 1.0


In [ ]:
def plot_fig1_style(selected_runs=None, traj: int = 0, include_init_column: bool = True):
    if selected_runs is None:
        selected_runs = paper_runs

    # Init from the first selected run. In your script, all runs usually use the same seed_init,
    # so this should be the same initialization across runs.
    init_run = selected_runs[0]
    init_W1 = get_W1(init_run, "init", traj=traj)
    init_G = gram_input_space(init_W1)

    Ws = []
    Gs = []
    titles = []

    if include_init_column:
        Ws.append(init_W1)
        Gs.append(init_G)
        titles.append("Initialization")

    for run in selected_runs:
        W = get_W1(run, "post", traj=traj)
        G = gram_input_space(W)
        loss = final_loss_for_run(run, traj=traj)
        Ws.append(W)
        Gs.append(G)
        titles.append(f"{short_paper_label(run)}\nloss={loss:.3g}")

    wlim = symmetric_abs_limit(Ws)
    glim = positive_limit(Gs)

    ncols = len(Ws)
    fig, axes = plt.subplots(2, ncols, figsize=(2.7 * ncols, 5.4), constrained_layout=True)

    if ncols == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for j, (W, G, title) in enumerate(zip(Ws, Gs, titles)):
        ax = axes[0, j]
        im = ax.imshow(W, aspect="auto", vmin=-wlim, vmax=wlim)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(r"Weight matrix $W_1$")

        ax = axes[1, j]
        ax.imshow(G, aspect="equal", vmin=0, vmax=glim)
        ax.set_xticks([])
        ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(r"Gram matrix $W_1^T W_1$")

    fig.suptitle("Figure 1 style: first-layer weights and input-space Gram matrices", y=1.03)
    plt.show()


# Recommended paper-like subset: GD, SGD batch sizes, and GD+weight-decay.
fig1_subset = [
    r for r in paper_runs
    if short_paper_label(r) in {"GD", "SGD-512", "SGD-32", "GD + wd"}
]

plot_fig1_style(fig1_subset, traj=0, include_init_column=True)


### Figure 1 style for all optimizers

This version includes Adam, AdamW, and Muon too. It is less paper-faithful but useful for your project extension.


In [ ]:
plot_fig1_style(paper_runs, traj=0, include_init_column=True)


## Figure 2 style: eigenvalue histograms for \(W_1\)

Strictly speaking, rectangular \(W_1\) does not have ordinary eigenvalues unless it is square. The paper's MNIST figure is best interpreted as an eigenvalue histogram of a Gram/covariance-type matrix derived from \(W_1\).

Here we use eigenvalues of:

\[
W_1^T W_1
\]

which are the squared singular values of \(W_1\). This is stable, always symmetric positive semidefinite, and directly tells us how much the first layer uses input-space directions.


In [ ]:
def gram_eigenvalues(W1: np.ndarray) -> np.ndarray:
    G = gram_input_space(W1)
    eigs = np.linalg.eigvalsh(G)
    eigs = np.clip(eigs, 0, None)
    return eigs


def plot_fig2_style(selected_runs=None, traj: int = 0, bins: int = 30, logy: bool = True):
    if selected_runs is None:
        selected_runs = paper_runs

    init_run = selected_runs[0]
    init_eigs = gram_eigenvalues(get_W1(init_run, "init", traj=traj))

    eig_lists = [init_eigs]
    titles = ["Initialization"]

    for run in selected_runs:
        eig_lists.append(gram_eigenvalues(get_W1(run, "post", traj=traj)))
        titles.append(short_paper_label(run))

    xmax = max(float(np.max(e)) for e in eig_lists)
    xmax = xmax if xmax > 0 else 1.0

    n = len(eig_lists)
    fig, axes = plt.subplots(1, n, figsize=(3.1 * n, 3.2), constrained_layout=True)

    if n == 1:
        axes = [axes]

    for ax, eigs, title in zip(axes, eig_lists, titles):
        ax.hist(eigs, bins=bins, range=(0, xmax))
        ax.axvline(np.median(eigs), linestyle="--", linewidth=1, label="median")
        ax.axvline(np.mean(eigs), linestyle=":", linewidth=1, label="mean")
        ax.set_title(title, fontsize=10)
        ax.set_xlabel(r"eigenvalue of $W_1^T W_1$")
        if logy:
            ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel("frequency")
    axes[-1].legend(fontsize=8)
    fig.suptitle("Figure 2 style: eigenvalue histograms of first-layer Gram matrix", y=1.08)
    plt.show()


plot_fig2_style(fig1_subset, traj=0, bins=20, logy=True)


## Figure 3 style: irrelevant norm over time

This is the closest match to the paper's theoretical plot.

The script already saved:

```python
np.linalg.norm(W1[:, r:d])
```

at every gradient step. So this figure is basically a paper-style reformatting of that curve.

The paper uses log-scaled x-axis to show the early and late regimes clearly.


In [ ]:
def median_and_quantiles(curves, q_low=0.25, q_high=0.75):
    min_len = min(len(c) for c in curves)
    arr = np.stack([c[:min_len] for c in curves])
    return (
        np.median(arr, axis=0),
        np.quantile(arr, q_low, axis=0),
        np.quantile(arr, q_high, axis=0),
    )


def plot_fig3_style(selected_runs=None, traj_mode: str = "median", max_points: int = 2500):
    if selected_runs is None:
        selected_runs = paper_runs

    plt.figure(figsize=(9, 5.5))

    for run in selected_runs:
        data = run["data"]
        curves = [data[f"traj{traj}_irelnorms"] for traj in run["traj_ids"]]

        if traj_mode == "median" and len(curves) > 1:
            y, lo, hi = median_and_quantiles(curves)
        else:
            y = curves[0]
            lo = hi = None

        n = len(y)
        # Start at 1 so log-x is valid.
        idx = np.unique(np.geomspace(1, n, min(max_points, n)).astype(int) - 1)
        x = idx + 1

        plt.plot(x, y[idx], label=short_paper_label(run))

        if lo is not None:
            plt.fill_between(x, lo[idx], hi[idx], alpha=0.15)

    plt.xscale("log")
    plt.xlabel("gradient step")
    plt.ylabel(r"$\|W_1[:, r:d]\|_F$")
    plt.title("Figure 3 style: norm of irrelevant first-layer weights over time")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_fig3_style(fig1_subset)


### Figure 3 style: all optimizers

This is useful for your extension, especially to compare Adam/AdamW/Muon against the GD/SGD behavior in the original paper.


In [ ]:
plot_fig3_style(paper_runs)


## Optional: separate Figure 3 by optimizer family

This avoids clutter if the all-optimizer plot is too dense.


In [ ]:
for family_name, predicate in {
    "GD / SGD / weight decay": lambda r: short_paper_label(r) in {"GD", "SGD-512", "SGD-32", "GD + wd"},
    "Adaptive / Muon": lambda r: r["config"].get("optimizer") in {"adam", "adamw", "muon"},
}.items():
    subset = [r for r in paper_runs if predicate(r)]
    if subset:
        print(family_name)
        plot_fig3_style(subset)
